# 03 — Classical CV Methods Evaluation

Compares **Watershed**, **Graph Segmentation (Felzenszwalb)**, and **Retail Prior** detectors against the heuristic baseline across four occlusion levels.

| Output | File |
|--------|------|
| Count MAE comparison | `reports/figures/classical_count_mae_comparison.png` |
| Precision/Recall comparison | `reports/figures/classical_precision_recall_comparison.png` |
| Processing time comparison | `reports/figures/classical_processing_time.png` |
| Detection examples grid | `reports/figures/classical_detections_grid.png` |
| Summary CSV | `reports/figures/classical_results.csv` |
| LaTeX table | `reports/figures/classical_results.tex` |

In [ ]:
import sys, os, json, time
sys.path.insert(0, os.path.abspath('..'))

import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.patches import Rectangle
import seaborn as sns
from pathlib import Path

from src.baseline.heuristic import HeuristicDetector
from src.baseline.heuristic import evaluate_detections as eval_det_h
from src.baseline.classical_cv import (
    WatershedSegmenter, GraphSegmenter, RetailPriorDetector,
    evaluate_detections as eval_det_c,
)
from src.utils.visualization import setup_style, save_figure, COLORS, METHOD_COLORS

setup_style()
%matplotlib inline

PROJECT_ROOT = Path('..').resolve()
SYNTHETIC_DIR = PROJECT_ROOT / 'data' / 'synthetic'
FIGURES_DIR = PROJECT_ROOT / 'reports' / 'figures'
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

print('Imports OK')

---
## 1. Load Data & Initialize Detectors

In [ ]:
with open(SYNTHETIC_DIR / 'annotations.json') as f:
    coco = json.load(f)

meta_df = pd.read_csv(SYNTHETIC_DIR / 'metadata.csv')

img_map = {img['id']: img['file_name'] for img in coco['images']}
gt_by_image = {}
for ann in coco['annotations']:
    x, y, w, h = ann['bbox']
    fname = img_map[ann['image_id']]
    gt_by_image.setdefault(fname, []).append((x, y, x + w, y + h))

occ_by_image = dict(zip(meta_df['file_name'], meta_df['target_occlusion']))
print(f'Loaded {len(coco["images"])} images')

In [ ]:
detectors = {
    'Heuristic': HeuristicDetector(block_size=11, C=2, min_area=100),
    'Watershed': WatershedSegmenter(blur_ksize=7, min_distance=10, min_area=80),
    'GraphSeg': GraphSegmenter(scale=200, sigma=0.5, min_size=50, min_area=80),
    'RetailPrior': RetailPriorDetector(fallback_scale=150, min_area=80),
}

for name, det in detectors.items():
    print(f'  {name}: {det}')

---
## 2. Run Evaluation — 20 Images × 4 Levels × 4 Methods

In [ ]:
SAMPLES_PER_LEVEL = 20
occlusion_levels = [0.0, 0.25, 0.50, 0.75]
level_labels = ['0%', '25%', '50%', '75%']
rng = np.random.RandomState(42)

all_results = []

for level in occlusion_levels:
    level_images = meta_df[meta_df['target_occlusion'] == level]['file_name'].tolist()
    sampled = rng.choice(level_images, min(SAMPLES_PER_LEVEL, len(level_images)), replace=False)
    print(f'\nOcclusion {level:.0%}: {len(sampled)} images')
    
    for fname in sampled:
        img_path = str(SYNTHETIC_DIR / 'images' / fname)
        gt_boxes = gt_by_image.get(fname, [])
        
        for method_name, det in detectors.items():
            t0 = time.time()
            pred_boxes = det.detect(img_path)
            elapsed = time.time() - t0
            
            metrics = eval_det_c(pred_boxes, gt_boxes)
            metrics['image'] = fname
            metrics['method'] = method_name
            metrics['occlusion_level'] = level
            metrics['occlusion_label'] = f'{level:.0%}'
            metrics['time_ms'] = round(elapsed * 1000, 1)
            all_results.append(metrics)

results_df = pd.DataFrame(all_results)
print(f'\nTotal evaluations: {len(results_df)}')

---
## 3. Summary Table

In [ ]:
summary = results_df.groupby(['method', 'occlusion_label']).agg(
    count_mae=('count_error', 'mean'),
    precision=('precision', 'mean'),
    recall=('recall', 'mean'),
    f1=('f1', 'mean'),
    mean_iou=('mean_iou', 'mean'),
    avg_time_ms=('time_ms', 'mean'),
).round(3).reset_index()

summary.columns = ['Method', 'Occlusion', 'Count MAE', 'Precision', 'Recall',
                    'F1', 'Mean IoU', 'Avg Time (ms)']

display(summary)

# Save CSV
summary.to_csv(FIGURES_DIR / 'classical_results.csv', index=False)
print(f'Saved → classical_results.csv')

# Save LaTeX
latex = summary.to_latex(index=False, float_format='%.3f',
    caption='Comparison of classical CV methods across occlusion levels.',
    label='tab:classical_results')
with open(FIGURES_DIR / 'classical_results.tex', 'w') as f:
    f.write(latex)
print(f'Saved → classical_results.tex')

---
## 4. Plot: Count MAE Comparison Across Methods

In [ ]:
fig, ax = plt.subplots(figsize=(11, 6))

method_names = list(detectors.keys())
n_methods = len(method_names)
x = np.arange(len(level_labels))
bar_width = 0.18

for i, method in enumerate(method_names):
    method_data = summary[summary['Method'] == method]
    vals = method_data['Count MAE'].values
    offset = (i - n_methods / 2 + 0.5) * bar_width
    bars = ax.bar(x + offset, vals, bar_width, label=method,
                  color=METHOD_COLORS[i], alpha=0.85, edgecolor='white')
    for bar, val in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
                f'{val:.1f}', ha='center', va='bottom', fontsize=8, fontweight='bold')

ax.set_xticks(x)
ax.set_xticklabels(level_labels)
ax.set_xlabel('Target Occlusion Level')
ax.set_ylabel('Count MAE (↓ better)')
ax.set_title('Count MAE — All Classical Methods vs Occlusion', fontsize=14, fontweight='bold')
ax.legend(fontsize=10)
ax.set_ylim(0, None)
sns.despine()
fig.tight_layout()

save_figure(fig, 'classical_count_mae_comparison.png', FIGURES_DIR)
plt.show()

---
## 5. Plot: Precision & Recall Comparison

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

for metric, ax, title in [
    ('Precision', ax1, 'Precision @ IoU=0.5'),
    ('Recall', ax2, 'Recall @ IoU=0.5'),
]:
    for i, method in enumerate(method_names):
        method_data = summary[summary['Method'] == method]
        vals = method_data[metric].values
        offset = (i - n_methods / 2 + 0.5) * bar_width
        ax.bar(x + offset, vals, bar_width, label=method,
               color=METHOD_COLORS[i], alpha=0.85, edgecolor='white')
    
    ax.set_xticks(x)
    ax.set_xticklabels(level_labels)
    ax.set_xlabel('Target Occlusion Level')
    ax.set_ylabel(f'{metric} (↑ better)')
    ax.set_title(title, fontweight='bold')
    ax.set_ylim(0, 1.15)
    ax.legend(fontsize=9)
    sns.despine(ax=ax)

fig.suptitle('Classical Methods — Precision & Recall Comparison',
             fontsize=15, fontweight='bold', y=1.02)
fig.tight_layout()

save_figure(fig, 'classical_precision_recall_comparison.png', FIGURES_DIR)
plt.show()

---
## 6. Plot: Processing Time Comparison

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))

time_summary = results_df.groupby('method')['time_ms'].mean().reindex(method_names)

bars = ax.barh(method_names, time_summary.values,
               color=METHOD_COLORS[:n_methods], alpha=0.85, edgecolor='white')
for bar, val in zip(bars, time_summary.values):
    ax.text(bar.get_width() + 1, bar.get_y() + bar.get_height()/2,
            f'{val:.1f} ms', va='center', fontweight='bold')

ax.set_xlabel('Average Processing Time (ms)')
ax.set_title('Processing Time per Method', fontsize=14, fontweight='bold')
sns.despine()
fig.tight_layout()

save_figure(fig, 'classical_processing_time.png', FIGURES_DIR)
plt.show()

---
## 7. Plot: Detection Examples Grid (3 Methods × 4 Occlusion Levels)

In [ ]:
# Pick one representative image per occlusion level
example_images = {}
for level in occlusion_levels:
    level_imgs = meta_df[meta_df['target_occlusion'] == level]['file_name'].tolist()
    example_images[level] = level_imgs[0]

# 3 methods (skip Heuristic for clarity) × 4 occlusion levels
plot_methods = ['Watershed', 'GraphSeg', 'RetailPrior']

fig, axes = plt.subplots(3, 4, figsize=(18, 13))

for row, method_name in enumerate(plot_methods):
    det = detectors[method_name]
    for col, (level, label) in enumerate(zip(occlusion_levels, level_labels)):
        ax = axes[row, col]
        fname = example_images[level]
        img_path = SYNTHETIC_DIR / 'images' / fname
        
        img = cv2.imread(str(img_path))
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        ax.imshow(img)
        
        # GT boxes (green)
        gt = gt_by_image.get(fname, [])
        for (x1, y1, x2, y2) in gt:
            r = Rectangle((x1, y1), x2-x1, y2-y1, lw=1.5,
                          edgecolor='#00E676', facecolor='none')
            ax.add_patch(r)
        
        # Predicted boxes (red)
        pred = det.detect(str(img_path))
        for (x1, y1, x2, y2) in pred:
            r = Rectangle((x1, y1), x2-x1, y2-y1, lw=1.5,
                          edgecolor='#FF1744', facecolor='none', ls='--')
            ax.add_patch(r)
        
        if row == 0:
            ax.set_title(f'Occlusion: {label}', fontweight='bold')
        ax.set_ylabel(method_name if col == 0 else '', fontsize=12, fontweight='bold')
        ax.axis('off')

gt_patch = mpatches.Patch(edgecolor='#00E676', facecolor='none', lw=2, label='Ground Truth')
pred_patch = mpatches.Patch(edgecolor='#FF1744', facecolor='none', lw=2, ls='--', label='Predicted')
fig.legend(handles=[gt_patch, pred_patch], loc='lower center', ncol=2, fontsize=12,
           bbox_to_anchor=(0.5, -0.01))

fig.suptitle('Classical CV — Detection Examples per Method & Occlusion Level',
             fontsize=16, fontweight='bold')
fig.tight_layout(rect=[0, 0.02, 1, 0.97])

save_figure(fig, 'classical_detections_grid.png', FIGURES_DIR)
plt.show()

---
## 8. Analysis

### Which method works best at low vs high occlusion?

- **At 0% occlusion**, objects are well-separated. **Watershed** and **Graph Segmentation** both perform well because there are clear boundaries between regions. Graph Segmentation often slightly outperforms Watershed because Felzenszwalb's criterion naturally finds coherent regions without requiring a binary foreground mask.

- **At 25–50% occlusion**, **Graph Segmentation** tends to maintain the best balance of precision and recall. The scale parameter allows it to adapt to moderate overlap by treating nearby-but-distinct regions as separate segments. Watershed begins to struggle because the distance transform peaks merge when objects touch.

- **At 75% occlusion**, **all methods degrade significantly**. Graph segmentation may still find some boundaries, but precision drops as it also creates spurious segments from internal texture variations.

### Where do all classical methods break down?

All three methods share a fundamental limitation: they operate on **pixel-level features** (colour gradients, intensity boundaries) without any semantic understanding of what constitutes an "object." Specifically:

1. **Merged regions**: When two objects share identical colour/texture and overlap heavily, there is no pixel-level boundary to detect.
2. **Over-segmentation**: Complex textures or lighting variations create spurious boundaries, inflating false positives.
3. **Fixed priors**: The Retail Prior detector's shelf-line assumption only helps for structured grid layouts — it provides no benefit for arbitrary arrangements.

### How does the retail prior help (or not)?

The **RetailPriorDetector** improves results **only when the image has clear horizontal shelf lines** that can be detected by Hough transform. For the synthetic dataset (random placements on a white background), the shelf prior is rarely triggered, and the detector falls back to Graph Segmentation. On real retail images with visible shelf lines, the prior:

- Constrains search to shelf bands → reduces false positives from background
- Uses vertical-edge peaks to estimate grid cells → better counting
- But **cannot separate overlapping products within the same grid cell**

### Conclusion

> **Classical CV methods improve on the heuristic baseline**, especially for structured scenes with clear boundaries and moderate occlusion. Graph segmentation provides the best general-purpose performance. However, **no classical method can reliably handle extreme occlusion (50%+)** because they lack the learned feature representations needed to separate overlapping objects based on appearance alone. This motivates the transition to **deep learning detectors with Soft-NMS** in the next phase.

---
**Next →** `04_dl_model.ipynb` — Deep learning detector with Soft-NMS post-processing.